# 02. Limpieza Editorial y Lematización del Corpus

Este notebook parte del dataset extraído por `src/extract_posteos_anfibia.py` (`data/processed/posteos_anfibia.parquet`).

## Objetivos:
1. Filtrar publicaciones sin cuerpo (`body_text` vacío: posteos de solo imágenes/podcast/redirecciones).
2. Descartar versiones y republicaciones anteriores identificadas con `-N/` en el slug.
3. Detectar idiomas con `langdetect` conservando español y casos asignados a portugués.
4. Construir `document_text_extended` integrando `copete` y `tagline`.
5. Ejecutar lematización en lote mediante spaCy (`es_core_news_md`), aplicando correcciones de lemas específicas (ej. "país").
6. Exportar `data/processed/corpus_anfibia_lematizado.parquet` y `.csv`.

In [5]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_colwidth", 120)

# Rutas estándar del proyecto
BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

PROCESSED_DATA_PATH = BASE_DIR / "data" / "processed"
DATA_PATH = PROCESSED_DATA_PATH / "posteos_anfibia.parquet"

assert DATA_PATH.exists(), f"Ejecutar primero: python src/extract_posteos_anfibia.py (Falta {DATA_PATH.name})"

df = pd.read_parquet(DATA_PATH)
df["post_date"] = pd.to_datetime(df["post_date"], errors="coerce") 

print(f"Posts cargados: {len(df):,}")
print(f"Columnas: {list(df.columns)}")

Posts cargados: 3,622
Columnas: ['ID', 'source_url', 'guid', 'post_name', 'post_title', 'title_text', 'post_status', 'post_type', 'post_author', 'post_date', 'post_date_gmt', 'post_modified', 'post_modified_gmt', 'post_date_calendar', 'post_year', 'post_month', 'post_year_month', 'post_quarter', 'post_year_quarter', 'post_weekday', 'days_to_modified', 'comment_count', 'post_excerpt', 'excerpt_text', 'post_content', 'body_text', 'document_text', 'word_count', 'copete', 'tagline', 'autor_ids', 'autor_id_list', 'autor_names', 'illustrator_ids', 'illustrator_id_list', 'illustrator_names', 'thumbnail_id', 'yoast_metadesc', 'yoast_focuskw', 'yoast_reading_time_minutes', 'category', 'destacado', 'destacado-arriba', 'destacado-medio', 'etiquetas-de-recomendados', 'home-slider', 'nav_menu', 'post_format', 'post_tag', 'recomendado', 'slider-en-home', 'spnl_log_type', 'super-destacado', 'text_type', 'tipo-de-cronica', 'web_story_category', 'web_story_media_source', 'web_story_tag', 'wp_pattern_ca

### 1. Posts con body_text vacío

El script de exploracion (`01_exploracion_y_diagnostico.ipynb`) reportó 26 posts sin cuerpo de texto. Hay que ver qué son antes de decidir si se descartan.

Repaso de las variables relevantes, considerándose "limpio" cuando el texto no contiene HTML, bloques de Gutenberg, shortcodes ni entidades HTML:
- `title_text`: es el titulo limpio 
- `body_text`: es el cuerpo del texto limpio
- `excerpt_text`: es el extracto limpio 
- `document_text`: es el conjunto de title_text + body_text
- `word_count`: cantidad de palabras de document_text
- `copete`: es un anticipo sobre que trata el post
- `tagline`: es una frase de presentacion

In [6]:
empty_body = df[df["body_text"].fillna("").str.strip().eq("")]
print(f"Posts con body_text vacío: {len(empty_body)}")
display(empty_body[["ID", "post_date", "title_text", "word_count", "source_url"]].sort_values("post_date"))

Posts con body_text vacío: 13


,ID,post_date,title_text,word_count,source_url
148,1755,2014-06-21 19:37:08,"Por lo menos así, lo veo yo",7,https://www.revistaanfibia.com/la-critica-anfibia/
274,14888,2014-10-31 10:59:18,El nadador,2,https://www.revistaanfibia.com/el-nadador/
431,24813,2015-06-06 05:56:40,"Venganza, judíos",2,https://www.revistaanfibia.com/venganza-judios/
564,30664,2015-10-09 07:43:01,Test peronista,2,https://www.revistaanfibia.com/test-peronista/
1404,94370,2018-01-29 11:18:53,El héroe del Mediterráneo,4,https://www.revistaanfibia.com/heroe-mediterraneo/
1549,103460,2018-06-06 16:59:47,La ciudad de las mujeres,5,https://www.revistaanfibia.com/la-ciudad-de-las-mujeres/
1554,103807,2018-06-11 10:45:48,La vida en verde,4,https://www.revistaanfibia.com/la-vida-en-verde/
1560,123696,2018-06-13 11:17:19,Desayuno con autoras,3,https://www.revistaanfibia.com/desayuno-con-autoras/
1566,123539,2018-06-15 11:51:14,Las figus de las pioneras,5,https://www.revistaanfibia.com/las-figus-de-las-pioneras/
1661,94860,2018-09-05 15:46:08,Gente que sí: Miguel Rep,5,https://www.revistaanfibia.com/gente-miguel-rep/


## 2. Detección de Idioma y Versiones Anteriores

In [3]:
# 1. Detección de idioma
def detect_lang(text):
    if not text or len(str(text).strip()) < 50:
        return None
    try:
        return detect(str(text)[:500])
    except LangDetectException:
        return None

df["lang"] = df["body_text"].map(detect_lang)

# 2. Identificación de republicaciones (-N/ en URL)
def slug_sin_sufijo(url):
    match = re.search(r"(https://www\.revistaanfibia\.com/.+)-\d+/$", str(url))
    return match.group(1) + "/" if match else None

versiones_mask = df["source_url"].str.contains(r"-\d+/", regex=True, na=False)
versiones_temp = df[versiones_mask].copy()
versiones_temp["url_original"] = versiones_temp["source_url"].map(slug_sin_sufijo)
urls_con_version_nueva = set(versiones_temp["url_original"].dropna())

ids_version_vieja = set(
    df.loc[df["source_url"].isin(urls_con_version_nueva), "ID"]
)
print(f"IDs excluidos por tener versión más nueva: {len(ids_version_vieja)}")

IDs excluidos por tener versión más nueva: 9


## 3. Construcción del Corpus Extendido y Filtrado

In [4]:
def build_document_text(row):
    parts = [str(row["title_text"]).strip()]
    copete = str(row.get("copete", "") or "").strip()
    if copete:
        parts.append(copete)
    tagline = str(row.get("tagline", "") or "").strip()
    if tagline:
        parts.append(tagline)
    body = str(row["body_text"]).strip()
    if body:
        parts.append(body)
    return ". ".join(p.rstrip(".") for p in parts if p)

df["document_text_extended"] = df.apply(build_document_text, axis=1)
df["word_count_extended"] = df["document_text_extended"].str.findall(r"(?u)\b\w+\b").str.len()

# Máscara de filtrado consolidada
mask = (
    df["body_text"].fillna("").str.strip().ne("")
    & ~df["document_text"].duplicated(keep="last")
    & df["lang"].isin(["es", "pt"])
    & ~df["ID"].isin(ids_version_vieja)
)

corpus = df[mask].copy().reset_index(drop=True)
corpus["document_text_final"] = corpus["document_text_extended"]
corpus.insert(0, "doc_id", corpus.index)

print(f"Corpus filtrado listo para lematizar: {len(corpus):,} documentos")

Corpus filtrado listo para lematizar: 3,556 documentos
